# 🚀 Data Science Project Notebook

## Project Defaults and Library Imports

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings 
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12,6)

pd.set_option('display.max_columns', 150)
pd.set_option('display.max_rows', 150)


# Dataset Imports, Shapes, and Common Features

In [0]:
train = pd.read_csv('../data/raw/application_train.csv')
test = pd.read_csv('../data/raw/application_test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\nTrain data has Target: {'TARGET' in train.columns}")
print(f"Test data has Target: {('TARGET' in test.columns)}")
print(f"\nCommon Features: {len(set(train.columns) & set(test.columns))}")


# Quick Look Inside Train Data

In [0]:
train.head(5)

# Target Counts and Distributions

In [0]:
target_counts = train['TARGET'].value_counts()
target_pct = train['TARGET'].value_counts(normalize=True) * 100

print('Target Distribution: ')
print(f"0 (No default): {target_counts[0]:,} ({target_pct[0]:.2f}%)")
print(f"1 (Default): {target_counts[1]:,} ({target_pct[1]:.2f}%)")

fig, ax = plt.subplots(1, 2, figsize=(12,4))

#Bar Chart for count
train['TARGET'].value_counts().plot(kind='bar', ax=ax[0], color=['green', 'red'])
ax[0].set_title('Target Distribution (Count)')
ax[0].set_xlabel('Target')
ax[0].set_ylabel('Count')
ax[0].set_xticklabels(['No Default', 'Default'], rotation=0)

# Pie chart for proportion
labels = ['No Default', 'Default']
colors = ['green', 'red']
explode = (0.05, 0.05)
ax[1].pie(train['TARGET'].value_counts().values, 
          labels=labels, 
          colors=colors, 
          explode=explode, 
          autopct='%1.2f%%', 
          startangle=90, 
          textprops={'fontsize': 10})

ax[1].set_title('Target Disribution (Proportion)')

plt.tight_layout()
plt.show()


In [0]:
print('Data Types:')
print(train.dtypes.value_counts())

# Column Analysis

In [0]:
def analyze_columns(df):
    analysis = pd.DataFrame({
        'dtype': df.dtypes,
        'null_pct' : (df.isnull().sum() / len(df) * 100).round(2),
        'unique_count' : df.nunique(),
        'unique_pct' : (df.nunique() / len(df) * 100).round(2)
    })

    analysis['category'] = 'numeric'
    analysis.loc[analysis['dtype'] == 'object', 'category'] = 'categorical'
    analysis.loc[analysis['unique_count'] == 1, 'category'] = 'constant'
    analysis.loc[analysis['unique_count'] == len(df), 'category'] = 'unique_id'

    return analysis

col_analysis = analyze_columns(train)
print('Column Analysis:')
col_analysis

## Missing Data

In [0]:
missing_stats = pd.DataFrame({
    'train_count' : len(train),
    'train_null_count' : train.isnull().sum(),
    'train_missing_pct' : (train.isnull().sum() / len(train) * 100).round(2),
    'test_count' : len(test),
    'test_null_count' : test.isnull().sum(),
    'test_missing_pct' : (test.isnull().sum() / len(test) * 100).round(2)
})
#Keeps only columns where train dataset has missing values.
missing_stats = missing_stats[missing_stats['train_missing_pct'] > 0]

#Sorts columns from highest missing % → lowest.
missing_stats = missing_stats.sort_values('train_missing_pct', ascending=False)

print(f"Train observations: {len(train):,} | Test observations: {len(test):,}\n")
missing_stats[['train_count', 'train_null_count', 'train_missing_pct', 'test_count', 'test_null_count', 'test_missing_pct']]

In [0]:
days_employed = train['DAYS_EMPLOYED']
q1 = days_employed.quantile(0.25)
q3 = days_employed.quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = train[(days_employed < lower_bound) | (days_employed > upper_bound)]
print(f"Number of outliers in DAYS_EMPLOYED: {len(outliers)}")
display(outliers[['DAYS_EMPLOYED']])

In [0]:
[col for col in train.columns if 'DAYS' in col]
[col for col in train.columns if 'AMT' in col]


In [0]:
age_features  = ['DAYS_BIRTH', 'DAYS_EMPLOYED']
amount_features = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE']
score_features = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
other_features = ['REGION_POPULATION_RELATIVE', 'OWN_CAR_AGE']

selected_features = age_features + amount_features + score_features + other_features
selected_features = [ feature for feature in selected_features if feature in train.columns and feature in test.columns]

train_plot = train[selected_features].copy()
test_plot = test[selected_features].copy()

feature_names_mapping = {} 

for col in age_features:
    if col in train_plot.columns:
        if col == 'DAYS_BIRTH':
            train_plot[col] = (-train_plot[col] / 365).round(1)
            test_plot[col] = (-test_plot[col] / 365).round(1)
            feature_names_mapping[col] = 'AGE (years)'

        elif col == 'DAYS_EMPLOYED':
            train_plot[col] = train_plot[col].replace(365243, np.nan)
            test_plot[col] = test_plot[col].replace(365243, np.nan)
            train_plot[col] = (-train_plot[col] / 365).round(1)
            test_plot[col] = (-test_plot[col] / 365).round(1)
            feature_names_mapping[col] = 'EMPLOYMENT (years)'


for col in amount_features:
    if col in train_plot.columns:
        train_plot[col] = train_plot[col] / 1000
        test_plot[col] = test_plot[col] / 1000
        feature_names_mapping[col] = col.replace('AMT_', '') + ' (K)'

for col in score_features + other_features:
    if col in train_plot.columns:
        feature_names_mapping[col] = col 
        